In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE = "/content/drive/MyDrive/VoxIntegrity_Data/ASVspoof2019_LA"

print("BASE exists:", os.path.exists(BASE))
print("\nContents:")
print(os.listdir(BASE))

BASE exists: True

Contents:
['asvspoof-2019-la-subset.zip', 'ASVspoof2019_LA_asv_protocols', 'ASVspoof2019_LA_asv_scores', 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019_LA_dev', 'ASVspoof2019_LA_eval', 'ASVspoof2019_LA_train (1)', 'ASVspoof2019_LA_train']


In [ ]:
TRAIN_DIR = os.path.join(BASE, "ASVspoof2019_LA_train", "flac")
DEV_DIR   = os.path.join(BASE, "ASVspoof2019_LA_dev", "flac")
EVAL_DIR  = os.path.join(BASE, "ASVspoof2019_LA_eval", "flac")

PROTOCOL_DIR = os.path.join(
    BASE, "ASVspoof2019_LA_cm_protocols"
)

TRAIN_PROTOCOL = os.path.join(
    PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"
)

DEV_PROTOCOL = os.path.join(
    PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"
)

EVAL_PROTOCOL = os.path.join(
    PROTOCOL_DIR, "ASVspoof2019.LA.cm.eval.trl.txt"
)

print("TRAIN:", TRAIN_DIR)
print("DEV  :", DEV_DIR)
print("EVAL :", EVAL_DIR)

print("\nAll folders exist:",
      all(os.path.exists(p) for p in [
          TRAIN_DIR, DEV_DIR, EVAL_DIR,
          TRAIN_PROTOCOL, DEV_PROTOCOL, EVAL_PROTOCOL
      ]))

TRAIN: /content/drive/MyDrive/VoxIntegrity_Data/ASVspoof2019_LA/ASVspoof2019_LA_train/flac
DEV  : /content/drive/MyDrive/VoxIntegrity_Data/ASVspoof2019_LA/ASVspoof2019_LA_dev/flac
EVAL : /content/drive/MyDrive/VoxIntegrity_Data/ASVspoof2019_LA/ASVspoof2019_LA_eval/flac

All folders exist: True


In [ ]:
import pandas as pd

train_rows = []

with open(TRAIN_PROTOCOL, "r") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) >= 5:
            train_rows.append({
                "audio_id": parts[1],
                "audio_path": os.path.join(
                    TRAIN_DIR, parts[1] + ".flac"
                ),
                "label": parts[4]
            })

train_df = pd.DataFrame(train_rows)

print("TRAIN samples:", len(train_df))
print("\nLabels:")
print(train_df["label"].value_counts())

print(
    "\nMissing files:",
    (~train_df["audio_path"].apply(os.path.exists)).sum()
)

TRAIN samples: 25380

Labels:
label
spoof       22800
bonafide     2580
Name: count, dtype: int64

Missing files: 0


In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_DIR = "/content/drive/MyDrive/VoxIntegrity_Data/models/AST-ASVspoof2019"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = model.to(device)

print("Model loaded successfully")
print("Labels:", model.config.id2label)

Device: cuda


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Model loaded successfully
Labels: {0: 'Bonafide', 1: 'Spoof'}


In [ ]:
from torch.utils.data import Dataset
import torch.nn.functional as F
import librosa

class ASVSpoofDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        waveform, sr = librosa.load(
            row["audio_path"],
            sr=16000,
            mono=True
        )

        max_length = 16000 * 4

        if len(waveform) > max_length:
            waveform = waveform[:max_length]

        else:
            waveform = F.pad(
                torch.tensor(waveform),
                (0, max_length - len(waveform))
            ).numpy()

        inputs = feature_extractor(
            waveform,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs["input_values"].squeeze(0)

        label = 0 if row["label"] == "bonafide" else 1

        return {
            "input_values": input_values,
            "labels": torch.tensor(label, dtype=torch.long)
        }


train_dataset = ASVSpoofDataset(train_df)

print("Dataset size:", len(train_dataset))

sample = train_dataset[0]

print("Input shape:", sample["input_values"].shape)
print("Label:", sample["labels"].item())

Dataset size: 25380
Input shape: torch.Size([1024, 128])
Label: 0


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

print("Batches:", len(train_loader))

batch = next(iter(train_loader))

print("Batch input shape:", batch["input_values"].shape)
print("Batch labels shape:", batch["labels"].shape)
print("Labels:", batch["labels"])

Batches: 6345
Batch input shape: torch.Size([4, 1024, 128])
Batch labels shape: torch.Size([4])
Labels: tensor([1, 1, 1, 1])


In [ ]:
from torch.optim import AdamW
import os

learning_rate = 1e-5

optimizer = AdamW(
    model.parameters(),
    lr=learning_rate
)

CHECKPOINT_DIR = "/content/drive/MyDrive/VoxIntegrity_Data/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Learning rate:", learning_rate)
print("Checkpoint folder:", CHECKPOINT_DIR)
print("Exists:", os.path.exists(CHECKPOINT_DIR))

Learning rate: 1e-05
Checkpoint folder: /content/drive/MyDrive/VoxIntegrity_Data/checkpoints
Exists: True


In [ ]:
import time
import torch

model.train()

num_epochs = 1
checkpoint_every = 500

total_batches = len(train_loader)

print("Starting full TRAIN fine-tuning")
print("Samples:", len(train_dataset))
print("Batches:", total_batches)
print("Epochs:", num_epochs)
print("Checkpoint every:", checkpoint_every, "batches")
print("Device:", device)

start_time = time.time()

for epoch in range(num_epochs):

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):

        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_values=input_values,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(outputs.logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        batch_num = batch_idx + 1

        if batch_num % 100 == 0:
            print(
                f"Batch {batch_num}/{total_batches} "
                f"| Loss: {loss.item():.4f} "
                f"| Accuracy: {correct/total:.4f}"
            )

        if batch_num % checkpoint_every == 0:

            checkpoint_path = os.path.join(
                CHECKPOINT_DIR,
                f"ast_epoch{epoch+1}_batch{batch_num}.pt"
            )

            torch.save({
                "epoch": epoch + 1,
                "batch": batch_num,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            }, checkpoint_path)

            print("CHECKPOINT SAVED:", checkpoint_path)

    epoch_loss = total_loss / total_batches
    epoch_accuracy = correct / total

    print("\n" + "=" * 60)
    print(f"EPOCH {epoch+1} COMPLETE")
    print(f"Loss: {epoch_loss:.4f}")
    print(f"Accuracy: {epoch_accuracy:.4f}")
    print(f"Time: {(time.time()-start_time)/60:.2f} minutes")
    print("=" * 60)

Starting full TRAIN fine-tuning
Samples: 25380
Batches: 6345
Epochs: 1
Checkpoint every: 500 batches
Device: cuda
Batch 100/6345 | Loss: 0.0000 | Accuracy: 0.9975
Batch 200/6345 | Loss: 0.0000 | Accuracy: 0.9988
Batch 300/6345 | Loss: 0.0000 | Accuracy: 0.9942
Batch 400/6345 | Loss: 0.0000 | Accuracy: 0.9956
Batch 500/6345 | Loss: 0.0001 | Accuracy: 0.9965
CHECKPOINT SAVED: /content/drive/MyDrive/VoxIntegrity_Data/checkpoints/ast_epoch1_batch500.pt
Batch 600/6345 | Loss: 0.0749 | Accuracy: 0.9967
Batch 700/6345 | Loss: 0.0008 | Accuracy: 0.9961
Batch 800/6345 | Loss: 0.0006 | Accuracy: 0.9966
Batch 900/6345 | Loss: 0.0000 | Accuracy: 0.9967
Batch 1000/6345 | Loss: 0.0001 | Accuracy: 0.9968
CHECKPOINT SAVED: /content/drive/MyDrive/VoxIntegrity_Data/checkpoints/ast_epoch1_batch1000.pt
Batch 1100/6345 | Loss: 0.0000 | Accuracy: 0.9968
Batch 1200/6345 | Loss: 0.0002 | Accuracy: 0.9971
Batch 1300/6345 | Loss: 0.0000 | Accuracy: 0.9973
Batch 1400/6345 | Loss: 0.0001 | Accuracy: 0.9975
Batch 